# 08 — XY-ring mixer with Dicke initial state

**Sweep 5.** Head-to-head benchmark of the standard transverse-field
mixer + uniform `|+>^n` start (the protocol used in notebooks 03–07)
against the **constraint-preserving XY ring mixer + Dicke initial state**
(introduced as exposition in the Methods section).

Problem: $n = 8$, $K = 2$ — small enough that exact diagonalisation
is instant and the XY mixer's sparse `expm_multiply` is cheap. We use
the canonical 16-equity universe restricted to the first 8 assets so
the $(\mu, \Sigma)$ is consistent with the other notebooks.

What we expect:
- **Standard mixer (penalty enforced)**: $P(\text{feasible}) < 1$, depends on $A$.
- **XY-Dicke**: $P(\text{feasible}) = 1$ identically; the dynamics never leaves the weight-$K$ subspace.
- **Quality**: XY-Dicke should match or beat the standard mixer at the same depth, because it explores only the relevant $\binom{n}{K}$-dimensional subspace.
- **Cost**: XY's `expm_multiply` is per-layer dominant; runtime is higher than the X-mixer's factorised single-qubit rotations.

> **Runtime:** ~3 min on Colab CPU. Run this **before** notebook 09 so
> the final comparison plots can read the `xy_comparison.json` cache.

In [ ]:
# === Bootstrap (Colab + local) ===
import os, urllib.request as _u
exec((open('../scripts/bootstrap.py') if os.path.exists('../scripts/bootstrap.py') else _u.urlopen('https://raw.githubusercontent.com/egil10/fys5419/main/project2/code/scripts/bootstrap.py')).read())

# === Project imports ===
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.colab     import out_dir
from scripts.data      import load_universe
from scripts.portfolio import PortfolioProblem, DEFAULTS
from scripts.classical import brute_force
from scripts.qaoa      import solve, solve_xy, make_hamiltonians
from scripts.metrics   import prob_optimal, prob_feasible
from scripts.plotting  import apply_style, PALETTE, title, fig_path
apply_style()

RESULTS = out_dir('results')
print(f'Results will be saved to: {RESULTS}')

In [ ]:
# === Small instance: n=8, K=2 ===
N_QUBITS = 8
K        = 2
P_VALUES = [1, 2, 3, 4]
N_RESTARTS = 20

r = load_universe()
idx = list(range(N_QUBITS))  # first 8 assets of the canonical universe
rsub = r.subset(idx)

pf = PortfolioProblem(rsub.mu, rsub.Sigma,
                      lam=DEFAULTS['lam'], A=DEFAULTS['A'], K=K,
                      tickers=rsub.tickers)
bf = brute_force(pf)
print(f'sub-universe: {rsub.tickers}')
print(f'brute force:  {bf.bitstring}  C={bf.cost:.6f}  picks={bf.tickers(pf)}')

In [ ]:
cache = RESULTS / 'xy_comparison.json'

if cache.exists():
    rows = json.loads(cache.read_text())
    print(f'loaded {cache.name}')
else:
    rows = []
    for p in P_VALUES:
        # Standard X-mixer + uniform start (the project's main solver).
        res_x  = solve(pf, p=p, n_restarts=N_RESTARTS, seed=42)
        # XY ring + Dicke (constraint-preserving).
        res_xy = solve_xy(pf, p=p, n_restarts=N_RESTARTS, seed=42)

        for tag, res in [('x_uniform', res_x), ('xy_dicke', res_xy)]:
            rows.append({
                'protocol':    tag,
                'p':           p,
                'energy':      float(res['energy']),
                'ratio':       float(res['ratio']),
                'p_optimal':   prob_optimal(res['probs'], bf.x),
                'p_feasible':  prob_feasible(res['probs'], pf.n, pf.K),
                'runtime_s':   float(res['runtime']),
            })
            print(f'  {tag:10s} p={p}: ratio={res["ratio"]:.4f}  '
                  f'P(opt)={prob_optimal(res["probs"], bf.x):.4f}  '
                  f'P(feas)={prob_feasible(res["probs"], pf.n, pf.K):.4f}  '
                  f'runtime={res["runtime"]:.2f}s')

    cache.write_text(json.dumps(rows, indent=2))
    print(f'saved -> {cache.name}')

pd.DataFrame(rows)

In [ ]:
df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for proto, color, marker in [('x_uniform', PALETTE['blue'],  'o'),
                              ('xy_dicke', PALETTE['ochre'], 's')]:
    sub = df[df.protocol == proto].sort_values('p')
    axes[0].plot(sub.p, sub.ratio,      f'{marker}-', color=color, lw=2, ms=8, label=proto)
    axes[1].plot(sub.p, sub.p_feasible, f'{marker}-', color=color, lw=2, ms=8, label=proto)
    axes[2].plot(sub.p, sub.runtime_s,  f'{marker}-', color=color, lw=2, ms=8, label=proto)

axes[0].axhline(1.0, color=PALETTE['red'], ls=':', lw=1.2, label='optimum')
axes[0].set_xticks(P_VALUES); axes[0].set_xlabel('depth p')
axes[0].set_ylabel('scaled ratio')
axes[0].set_ylim(0.5, 1.05)
title(axes[0], 'Approximation quality vs depth', 'X+uniform vs XY+Dicke at n=8, K=2')
axes[0].legend(fontsize=8); axes[0].grid(False)

axes[1].axhline(1.0, color=PALETTE['red'], ls=':', lw=1.2, label='feasible = 1')
axes[1].set_xticks(P_VALUES); axes[1].set_xlabel('depth p')
axes[1].set_ylabel('P(feasible)')
axes[1].set_ylim(0.0, 1.05)
title(axes[1], 'Feasibility', 'XY+Dicke = 1.0 by construction; X+uniform = 1.0 only in expectation')
axes[1].legend(fontsize=8); axes[1].grid(False)

axes[2].set_yscale('log')
axes[2].set_xticks(P_VALUES); axes[2].set_xlabel('depth p')
axes[2].set_ylabel('runtime (s, log)')
title(axes[2], 'Runtime per solve', 'XY mixer uses expm_multiply per layer; X is a factorised rotation')
axes[2].legend(fontsize=8); axes[2].grid(False)

plt.tight_layout()
fig.savefig(fig_path('compare', 'xy_mixer'), bbox_inches='tight')
plt.show()

### Sanity: $[H_M^{XY}, N] = 0$ on a random superposition

If the XY ring mixer truly commutes with the number operator $N = \sum_i x_i$, then
$\| H_M^{XY} N \, |\psi\rangle - N \, H_M^{XY} \, |\psi\rangle \| = 0$ for any $\psi$.
This is the operational check that the XY ring stays inside the
weight-$K$ subspace — if this fails, every other plot in the notebook is
worthless.

In [ ]:
from scripts.xy_mixer import build_HM_xy, number_operator_diag

n = pf.n
H_xy = build_HM_xy(n, periodic=True)
N_diag = number_operator_diag(n)

rng = np.random.default_rng(0)
psi = rng.normal(size=1 << n) + 1j * rng.normal(size=1 << n)
psi = psi / np.linalg.norm(psi)

lhs = H_xy @ (N_diag * psi)
rhs = N_diag * (H_xy @ psi)
commutator_norm = float(np.linalg.norm(lhs - rhs))
print(f'|| [H_M^XY, N] psi || = {commutator_norm:.2e}   (should be ~ machine epsilon)')

# Round-trip: dynamics from the Dicke state should preserve <N> = K exactly.
from scripts.dicke import dicke_state
dicke = dicke_state(n, pf.K)
exp_N_before = float(np.real(dicke.conj() @ (N_diag * dicke)))

from scripts.xy_mixer import apply_xy_mixer
psi_t = apply_xy_mixer(dicke, beta=0.37, H_M_xy=H_xy)
exp_N_after = float(np.real(psi_t.conj() @ (N_diag * psi_t)))
print(f'<N> on |D^{n}_{pf.K}>     = {exp_N_before:.6f}  (should be {pf.K})')
print(f'<N> after U_M(0.37)   = {exp_N_after:.6f}  (should still be {pf.K})')